# Lesson 6 — The transformer block

The repeatable unit of a GPT. A block = masked multi-head attention + a small feed-forward net, each wrapped in a
residual connection with a layer norm in front (pre-norm):

```
x = x + attention(norm(x))
x = x + feedforward(norm(x))
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

## Same embedded batch as lessons 4–5

In [2]:
# --- rebuild the embedded batch from Lessons 2-3 ---
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
block_size, batch_size, n_embd = 8, 4, 32

def get_batch(src, block, bs):
    ix = torch.randint(len(src) - block, (bs,))
    return (torch.stack([src[i:i + block] for i in ix]),
            torch.stack([src[i + 1:i + block + 1] for i in ix]))

xb, yb = get_batch(data, block_size, batch_size)
x = nn.Embedding(vocab_size, n_embd)(xb) + nn.Embedding(block_size, n_embd)(torch.arange(block_size))
B, T, C = x.shape

## 1. One masked head as a module
Lessons 4 + 5 packed into an `nn.Module`. The `tril` mask is a *buffer*: it moves with the model but isn't trained.
`tril[:T, :T]` lets it work with sequences shorter than `block_size`.

In [3]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

## 2. Multi-head attention
Several smaller heads run side by side, each free to look for a different kind of relationship. Their outputs are
concatenated and a linear `proj` mixes them back to `n_embd`.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

## 3. Feed-forward
Attention moves information *between* tokens; this MLP then processes each token on its own
(widen ×4 → ReLU → back to `n_embd`).

In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

## 4. The block: residuals + pre-norm
- `x + sublayer(...)` — the residual path lets gradients flow straight through, so each layer only learns a correction.
- `LayerNorm` before each sublayer keeps the activations at a sensible scale.

In [6]:
class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # residual around pre-normed attention
        x = x + self.ff(self.ln2(x))    # residual around pre-normed feed-forward
        return x

In [7]:
block = Block(n_embd, n_heads=4)
out = block(x)
print("block input :", tuple(x.shape))
print("block output:", tuple(out.shape), " <- same shape in and out, so blocks stack")
print("parameters in one block:", sum(p.numel() for p in block.parameters()))

block input : (4, 8, 32)
block output: (4, 8, 32)  <- same shape in and out, so blocks stack
parameters in one block: 12608


## Reference check
Course: input `(4, 8, 32)`, output `(4, 8, 32)`, **12608** parameters — mine matches. Below I break the 12608 down
by hand to see where the parameters live (4 heads × 8 dims each).

In [8]:
hs = n_embd // 4
by_hand = {
    "4 heads (k,q,v, no bias)": 4 * 3 * n_embd * hs,
    "proj (32*32 + 32)":        n_embd * n_embd + n_embd,
    "ff up (32*128 + 128)":     n_embd * 4 * n_embd + 4 * n_embd,
    "ff down (128*32 + 32)":    4 * n_embd * n_embd + n_embd,
    "2 layernorms (2*(32+32))": 2 * 2 * n_embd,
}
for k_, v_ in by_hand.items(): print(f"{k_:28s} {v_:6d}")
print(f"{'total':28s} {sum(by_hand.values()):6d}  == 12608: {sum(by_hand.values()) == 12608}")
print("output matches input shape:", out.shape == x.shape)

4 heads (k,q,v, no bias)       3072
proj (32*32 + 32)              1056
ff up (32*128 + 128)           4224
ff down (128*32 + 32)          4128
2 layernorms (2*(32+32))        128
total                         12608  == 12608: True
output matches input shape: True


## Test yourself

**1. Why must a block's output shape equal its input shape?**
So blocks can be stacked: block 2 takes block 1's output exactly as if it were the embeddings. A GPT is just N of
these in a row.

**2. What does the residual `x + sublayer(x)` buy us?**
A direct path for gradients through a deep stack, and each sublayer only has to learn a small change to `x` instead
of rebuilding it from scratch.

**3. Attention vs feed-forward — what does each do?**
Attention lets tokens exchange information (communication). Feed-forward works on each token separately to process
what it gathered (computation).

## My notes
- A block is "communicate, then think", each step added on top of `x` rather than replacing it.
- Surprising: two-thirds of the block's 12608 parameters are in the feed-forward net, not the attention.